In [1]:
import pickle

with open("model.pkl","rb") as f:
    model=pickle.load(f)

In [2]:
import cv2
import pandas as pd
import mediapipe as mp

mp_hands=mp.solutions.hands
hand=mp_hands.Hands()
mp_draw=mp.solutions.drawing_utils

In [3]:
cap = cv2.VideoCapture(0)
print("press esc to exit")

word = ""
current_prediction = ""
prev_prediction = ""
count = 0
inserted = False  

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hand.process(rgb)

    data = []
    prediction = ""

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            for lm in hand_landmarks.landmark:
                data.extend([lm.x, lm.y, lm.z])

        if len(data) == 63:
            prediction = model.predict([data])[0]

            if prediction == prev_prediction:
                count += 1
            else:
                count = 0
                inserted = False  

            if count > 10 and not inserted:
                if prediction == "space":
                    word += " "
                    current_prediction = " "
                    inserted = True

                elif prediction == "del":
                    if current_prediction != "":
                        current_prediction = ""
                    else:
                        word = word[:-1]
                    inserted = True

                elif prediction == "next":
                    word += current_prediction
                    current_prediction = ""
                    inserted = True

                else:
                    current_prediction = prediction
                    inserted = True  
                    
            prev_prediction = prediction

            cv2.putText(frame,
                f"Sign: {prediction}  ({'stable' if count > 10 else 'detecting'})",
                (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2)

            cv2.putText(frame,
                f"Letter: {current_prediction}",
                (20, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 165, 255),
                2)

            cv2.putText(frame,
                f"Word: {word}",
                (20, 150),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 0, 0),
                2)

    cv2.imshow("Sign Language Detection", frame)
    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()

press esc to exit


c:\Users\jiyam\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\jiyam\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\jiyam\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabas